## 1. Environment Setup and Experiment Import

This cell sets `KMP_DUPLICATE_LIB_OK` to avoid duplicate OpenMP runtime errors that can occur on some macOS/Intel environments. It then imports `experiment_mlp_gn_coverage` from `experiments.py`. This function builds the MLP objectives, runs the uniform discretisation baseline and/or the adaptive bundle method, and saves the resulting plot as a PNG file.

In [8]:
import os

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

from experiments import experiment_mlp_gn_coverage

## 2. Run the MLP Baseline and Generate a Plot

This cell runs the GN* coverage experiment for the non-convex MLP. It uses `K=3, p=10, n=20, h=8`, so the parameter dimension is `d = h*p + h + K*h + K = 115`. The value of `coarse_resolution=9/26/85` corresponds to `r=9/26/85` in the plot legend. Setting `run_baseline=True` and `run_adaptive=False` means that only the uniform discretisation baseline is plotted, without the adaptive bundle curve. The plot is saved as `mlp_K={K}_p={p}_n={n}_h={h}_err={err_label}(r={coarse_resolution}).png`.

In [2]:
K = 3
p = 10
n = 20
h = 8
seed = 10
coarse_resolution = 85
err_label = "1e-2"

result = experiment_mlp_gn_coverage(
    verbose=True,
    K=K,
    p=p,
    n=n,
    h=h,
    seed=seed,
    coarse_resolution=coarse_resolution,
    n_passes=15,
    steps_per_point_per_pass=50,
    eval_every_n_grads=5000,
    run_baseline=True,
    run_adaptive=False,
    out_path=f"mlp_K={K}_p={p}_n={n}_h={h}_err={err_label}(r={coarse_resolution}).png",
)

result["plot"]

Coverage experiment — MLP (non-convex), metric = GN*
  K=3, p=10, n=20, h=8, d=115  |  L=[3.613 3.739 2.503] 

  [uniform discretisation] ...
  Baseline pass 0/15 | t=0.00s | iters=0 | grad_evals=0 | worst-case pc=6.6667e-01
  Baseline pass 1/15 | t=17.73s | iters=187050 | grad_evals=561150 | worst-case pc=1.1588e-03
  Baseline pass 2/15 | t=32.84s | iters=374100 | grad_evals=1122300 | worst-case pc=8.5768e-04
  Baseline pass 3/15 | t=48.26s | iters=561150 | grad_evals=1683450 | worst-case pc=4.4375e-04
  Baseline pass 4/15 | t=63.47s | iters=748200 | grad_evals=2244600 | worst-case pc=2.6154e-04
  Baseline pass 5/15 | t=78.53s | iters=935250 | grad_evals=2805750 | worst-case pc=1.8807e-04
  Baseline pass 6/15 | t=93.75s | iters=1122300 | grad_evals=3366900 | worst-case pc=1.4368e-04
  Baseline pass 7/15 | t=108.90s | iters=1309350 | grad_evals=3928050 | worst-case pc=1.4558e-04
  Baseline pass 8/15 | t=124.05s | iters=1496400 | grad_evals=4489200 | worst-case pc=1.4583e-04
  Baseline 

'mlp_K=3_p=10_n=20_h=8_err=1e-2(r=85).png'

## 3. Compare the Two Algorithms at K=4, p=15

This module fixes a single MLP problem: `K=4, p=15, n=20, h=8, seed=10`. It runs both algorithms on this same problem: the uniform discretisation baseline and the adaptive bundle method. The algorithm budget follows the default settings used to generate `mlp.png`; only the model parameters `K` and `p` are changed from `K=3, p=10` to `K=4, p=15`.

In [ ]:
K = 3
p = 10
n = 20
h = 8
seed = 10

coarse_resolution = 26
n_passes = 15
steps_per_point_per_pass = 50
eval_every_n_grads = 5000
max_outer = 1000

comparison = experiment_mlp_gn_coverage(
    verbose=True,
    K=K,
    p=p,
    n=n,
    h=h,
    seed=seed,
    coarse_resolution=coarse_resolution,
    n_passes=n_passes,
    steps_per_point_per_pass=steps_per_point_per_pass,
    eval_every_n_grads=eval_every_n_grads,
    max_outer=max_outer,
    lambda_selector="sample",
    lambda_random_starts=512,
    run_baseline=True,
    run_adaptive=True,
    out_path=f"mlp_compare_K{K}_p{p}_n{n}_h{h}(r={coarse_resolution})_baseline_vs_adaptive.png",
)

comparison["plot"]

Coverage experiment — MLP (non-convex), metric = GN*
  K=3, p=10, n=20, h=8, d=115  |  L=[3.613 3.739 2.503] 

  [uniform discretisation] ...
  Baseline pass 0/15 | t=0.00s | iters=0 | grad_evals=0 | worst-case pc=6.6667e-01
  Baseline pass 1/15 | t=1.65s | iters=18900 | grad_evals=56700 | worst-case pc=3.3144e-03
  Baseline pass 2/15 | t=3.21s | iters=37800 | grad_evals=113400 | worst-case pc=1.6419e-03
  Baseline pass 3/15 | t=4.84s | iters=56700 | grad_evals=170100 | worst-case pc=1.6663e-03
  Baseline pass 4/15 | t=6.50s | iters=75600 | grad_evals=226800 | worst-case pc=1.0629e-03
  Baseline pass 5/15 | t=8.09s | iters=94500 | grad_evals=283500 | worst-case pc=9.8619e-04
  Baseline pass 6/15 | t=9.66s | iters=113400 | grad_evals=340200 | worst-case pc=9.8619e-04
  Baseline pass 7/15 | t=11.32s | iters=132300 | grad_evals=396900 | worst-case pc=9.8619e-04
  Baseline pass 8/15 | t=12.87s | iters=151200 | grad_evals=453600 | worst-case pc=9.8619e-04
  Baseline pass 9/15 | t=14.46s | i

'mlp_compare_K3_p10_n20_h8(r=26)_baseline_vs_adaptive.png'

: 

## 4. Increase the Adaptive Budget and Use the Baseline Final GN* as the Stopping Target

The plot in Section 3 shows that the baseline reaches a relatively low final GN*, while the adaptive method does not reach the baseline final GN* under the current parameters and budget. The red baseline curve ends at about `1.479e-03`, while the blue adaptive curve remains above that level within the `max_outer=1000` budget. This means the adaptive method stops because it hits the `max_outer` limit, not because it reaches the target.

If the baseline final error is very low, the adaptive method may require a much larger budget, and it may be difficult to reach that target under the current implementation or parameter choices. The conclusion should therefore be stated as: under the current adaptive parameters and budget, the adaptive method did not reach the baseline final GN*; increasing `max_outer` or adjusting the lambda-selection strategy, such as `lambda_selector` or `lambda_random_starts`, may be necessary.

The following experiment keeps the same `K=4, p=15, n=20, h=8, seed=10` problem and baseline settings, but increases the adaptive budget to `max_outer=5000, max_inner=50` and increases the number of lambda samples to `lambda_random_starts=2048`. Inside `experiment_mlp_gn_coverage`, the baseline final GN* is automatically used as the adaptive method's `target_cov`. The adaptive method stops early if it reaches this target; otherwise, it runs until `max_outer=5000`.

In [ ]:
K = 6
p = 15
n = 20
h = 8
seed = 10

coarse_resolution = 26
n_passes = 15
steps_per_point_per_pass = 50
eval_every_n_grads = 5000

max_outer = 5000
max_inner = 50
lambda_random_starts = 2048

comparison_larger_adaptive_budget = experiment_mlp_gn_coverage(
    verbose=True,
    K=K,
    p=p,
    n=n,
    h=h,
    seed=seed,
    coarse_resolution=coarse_resolution,
    n_passes=n_passes,
    steps_per_point_per_pass=steps_per_point_per_pass,
    eval_every_n_grads=eval_every_n_grads,
    max_outer=max_outer,
    max_inner=max_inner,
    lambda_selector="sample",
    lambda_random_starts=lambda_random_starts,
    run_baseline=True,
    run_adaptive=True,
    out_path=(
        f"mlp_compare_K{K}_p{p}_n{n}_h{h}"
        f"(r={coarse_resolution})_adaptive_outer{max_outer}_sample{lambda_random_starts}.png"
    ),
)

comparison_larger_adaptive_budget["plot"]

Coverage experiment — MLP (non-convex), metric = GN*
  K=5, p=15, n=20, h=8, d=173  |  L=[4.044 2.599 2.988 5.058 4.908] 

  [uniform discretisation] ...
  Baseline pass 0/15 | t=0.00s | iters=0 | grad_evals=0 | worst-case pc=8.0000e-01
  Baseline pass 1/15 | t=180.89s | iters=1370250 | grad_evals=6851250 | worst-case pc=4.8469e-03
  Baseline pass 2/15 | t=362.49s | iters=2740500 | grad_evals=13702500 | worst-case pc=2.6812e-03
  Baseline pass 3/15 | t=543.00s | iters=4110750 | grad_evals=20553750 | worst-case pc=1.9685e-03
  Baseline pass 4/15 | t=723.05s | iters=5481000 | grad_evals=27405000 | worst-case pc=1.9222e-03
  Baseline pass 5/15 | t=3698.41s | iters=6851250 | grad_evals=34256250 | worst-case pc=1.9589e-03
  Baseline pass 6/15 | t=4277.22s | iters=8221500 | grad_evals=41107500 | worst-case pc=1.8761e-03
  Baseline pass 7/15 | t=4459.70s | iters=9591750 | grad_evals=47958750 | worst-case pc=1.9228e-03


KeyboardInterrupt: 

## 7. Compare the Two Algorithms at K=4, p=15 with IPOPT Lambda Selection

This section repeats the Section 3 framework at the intended problem size — `K=4, p=15, n=20, h=8, seed=10` — and replaces the random-sampling lambda selector with the IPOPT-backed optimiser.

**What changes relative to Section 3:**

| Parameter | Section 3 | Section 7 |
|-----------|-----------|-----------|
| K, p | K=3, p=10 | **K=4, p=15** |
| `lambda_selector` | `"sample"` (512 random Dirichlet draws) | **`"optimize"` (IPOPT multi-start NLP)** |

All other algorithm parameters are kept identical to Section 3: `coarse_resolution=26`, `n_passes=15`, `steps_per_point_per_pass=50`, `eval_every_n_grads=5000`, `max_outer=1000`.

**Why IPOPT instead of random sampling:**

Each outer iteration of the adaptive method must solve
```
λ* = argmax_{λ ∈ Δ_K}  GN(λ; B)  =  max_{λ ∈ Δ_K}  min_i ‖J_F(xᵢ)ᵀ λ‖²
```
This is a non-concave maximisation over the K-simplex. The `"sample"` selector approximates it by evaluating GN at 512 random Dirichlet draws and returning the best. The `"optimize"` selector instead runs IPOPT (an interior-point NLP solver) from a structured set of starting points — simplex vertices, edge midpoints, the centroid, and the previous iterate — and refines each start with exact first-order information (analytical Danskin gradient). As K grows, this structured search is more reliable at escaping flat or misleading regions of the simplex that uniform random draws can miss.

**Expected behaviour:**

- Each outer iteration is slower (multiple IPOPT solves vs. a vectorised random sweep), so wall-clock time per outer step increases.
- The λ* quality is higher, meaning the adaptive method identifies harder directions more reliably and the bundle grows in a more targeted way.
- At the same `grad_evals` budget, the IPOPT curve should reach a lower GN* than the `"sample"` curve from Section 3, at the cost of higher wall-clock time per gradient evaluation.

In [4]:
K = 3
p = 10
n = 20
h = 8
seed = 10

coarse_resolution = 26
n_passes = 15
steps_per_point_per_pass = 50
eval_every_n_grads = 5000
max_outer = 1000

comparison_ipopt = experiment_mlp_gn_coverage(
    verbose=True,
    K=K,
    p=p,
    n=n,
    h=h,
    seed=seed,
    coarse_resolution=coarse_resolution,
    n_passes=n_passes,
    steps_per_point_per_pass=steps_per_point_per_pass,
    eval_every_n_grads=eval_every_n_grads,
    max_outer=max_outer,
    lambda_selector="optimize",
    run_baseline=True,
    run_adaptive=True,
    out_path=(
        f"mlp_compare_K{K}_p{p}_n{n}_h{h}"
        f"(r={coarse_resolution})_ipopt.png"
    ),
)

comparison_ipopt["plot"]

Coverage experiment — MLP (non-convex), metric = GN*
  K=3, p=10, n=20, h=8, d=115  |  L=[3.613 3.739 2.503] 

  [uniform discretisation] ...
  Baseline pass 0/15 | t=0.00s | iters=0 | grad_evals=0 | worst-case pc=6.6667e-01
  Baseline pass 1/15 | t=1.64s | iters=18900 | grad_evals=56700 | worst-case pc=3.5615e-03
  Baseline pass 2/15 | t=3.22s | iters=37800 | grad_evals=113400 | worst-case pc=1.7124e-03
  Baseline pass 3/15 | t=4.81s | iters=56700 | grad_evals=170100 | worst-case pc=1.6663e-03
  Baseline pass 4/15 | t=6.39s | iters=75600 | grad_evals=226800 | worst-case pc=1.0629e-03
  Baseline pass 5/15 | t=7.92s | iters=94500 | grad_evals=283500 | worst-case pc=9.8619e-04
  Baseline pass 6/15 | t=9.44s | iters=113400 | grad_evals=340200 | worst-case pc=9.8622e-04
  Baseline pass 7/15 | t=10.98s | iters=132300 | grad_evals=396900 | worst-case pc=9.8619e-04
  Baseline pass 8/15 | t=12.57s | iters=151200 | grad_evals=453600 | worst-case pc=9.8619e-04
  Baseline pass 9/15 | t=14.10s | i

'mlp_compare_K3_p10_n20_h8(r=26)_ipopt.png'

## 8. Increase the Adaptive Budget for the IPOPT Run and Use the Baseline Final GN* as the Stopping Target

Section 7 shows that the adaptive method with `lambda_selector="optimize"` (IPOPT) does not converge within `max_outer=1000`. This section follows the same approach as Section 4: increase the outer budget and let the baseline final GN* serve as the early-stopping target.

**What changes relative to Section 7:**

| Parameter | Section 7 | Section 8 |
|-----------|-----------|-----------|
| `max_outer` | 1000 | *5000** |
| `max_inner` | 50 (auto for K=4) | 50 (same) |

All other parameters are identical to Section 7: `K=4, p=15, n=20, h=8, seed=10`, `coarse_resolution=26`, `lambda_selector="optimize"`.

**How the stopping target works:**

Inside `experiment_mlp_gn_coverage`, the baseline is always run first. Its final GN* is automatically passed as `target_cov` to `algorithm_adaptive`. The adaptive method stops as soon as its checkpoint GN* drops to or below this target — or after `max_outer=3000` outer iterations if the target is never reached.

In [ ]:
K = 6
p = 5
n = 25
h = 8
seed = 10

coarse_resolution = 9
n_passes = 15
steps_per_point_per_pass = 50
eval_every_n_grads = 5000
max_outer = 100000
max_inner = 50

comparison_ipopt_larger_budget = experiment_mlp_gn_coverage(
    verbose=True,
    K=K,
    p=p,
    n=n,
    h=h,
    seed=seed,
    coarse_resolution=coarse_resolution,
    n_passes=n_passes,
    steps_per_point_per_pass=steps_per_point_per_pass,
    eval_every_n_grads=eval_every_n_grads,
    max_outer=max_outer,
    max_inner=max_inner,
    lambda_selector="optimize",
    run_baseline=True,
    run_adaptive=True,
    out_path=(
        f"mlp_compare_K{K}_p{p}_n{n}_h{h}"
        f"(r={coarse_resolution})_ipopt_outer{max_outer}.png"
    ),
)

comparison_ipopt_larger_budget["plot"]

Coverage experiment — MLP (non-convex), metric = GN*
  K=6, p=5, n=25, h=8, d=102  |  L=[2.59  3.918 2.263 2.495 2.394 3.758] 

  [uniform discretisation] ...
  Baseline pass 0/15 | t=0.00s | iters=0 | grad_evals=0 | worst-case pc=8.3333e-01
  Baseline pass 1/15 | t=14.85s | iters=100100 | grad_evals=600600 | worst-case pc=1.8651e-02
  Baseline pass 2/15 | t=29.88s | iters=200200 | grad_evals=1201200 | worst-case pc=1.8537e-02


## 10. Sample vs IPOPT Lambda Selector at K=8 — Head-to-Head

This section directly compares the two outer-loop λ selectors on the same K=8 problem.
Both methods run the same inner-loop budget (`max_inner=25`) and the same initial point `x₀=0`;
the only difference is how each outer iteration chooses the next λ to target.

**Problem:** K=8, p=10, n=40, h=8, seed=10 (d=160).  
n is raised to 40 (≥ K×5) to ensure every class has enough samples for a non-zero Lipschitz estimate —
with K=8 and n=20, random label assignment can leave some classes empty, causing L[i]=0 and ZeroDivisionError.

**Methods compared:**

| | Sample | IPOPT |
|--|--|--|
| λ selection | 512 random Dirichlet draws + structured starts | multi-start IPOPT NLP, 20 starts |
| Cost per outer | cheap (vectorised sweep, ~0.01 s) | expensive (20 × interior-point solves) |
| `max_outer` | 500 | 200 |
| Expected runtime | < 5 min | ~20–40 min |

A coarse baseline (r=3, only 120 simplex grid points) is run first solely to produce the green
reference line; it is not the main comparison target.

**What to look for:**
- **Right panel (grad_evals):** same inner budget per outer step, so the gap between curves
  shows whether IPOPT's more precise λ\* translates to faster GN\* decay per gradient evaluation.
- **Left panel (CPU time):** wall-clock tradeoff — sample completes many more outer iterations
  in the same time, while IPOPT spends more per iteration finding a harder direction.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from objectives import make_mlp_nonconvex
from baseline import uniform_discretisation
from algorithm import algorithm_adaptive

K = 8
p = 10
n = 40           # K*5 — avoids L[i]=0 when a class gets 0 samples
h = 8
seed = 10

coarse_resolution = 3   # C(10,7)=120 grid points — fast reference for K=8
n_passes = 15
steps_per_point_per_pass = 50
eval_every_n_grads = 2000
max_inner = 25           # default for K != 4

max_outer_sample = 500
lambda_random_starts = 512

max_outer_ipopt = 200
lambda_max_starts_ipopt = 20

# ── Build problem ──────────────────────────────────────────────────────────
d = h * p + h + K * h + K
objs, grads, L, joint = make_mlp_nonconvex(K=K, p=p, n=n, h=h, seed=seed)
x0 = np.zeros(d)
print(f"K={K}, p={p}, n={n}, h={h}, d={d}  |  L={np.round(L, 3)}")

# ── Baseline (coarse, for reference line only) ─────────────────────────────
print("\n[baseline — coarse reference r=3] ...")
bl = uniform_discretisation(
    K=K, objectives=objs, grad_objectives=grads, L=L, x0=x0,
    resolution=coarse_resolution, n_passes=n_passes,
    steps_per_point_per_pass=steps_per_point_per_pass,
    eval_every_n_grads=eval_every_n_grads,
    coverage_mode="gn", joint_oracle=joint, verbose=True)

# ── Adaptive: sample ───────────────────────────────────────────────────────
print("\n[adaptive — sample (512 draws)] ...")
a_sample = algorithm_adaptive(
    K=K, d=d, objectives=objs, grad_objectives=grads, L=L, x0=x0,
    mode="gn",
    max_outer=max_outer_sample, max_inner=max_inner,
    eval_every_n_grads=eval_every_n_grads,
    target_cov=bl["cov_history"][-1],
    lambda_selector="sample",
    lambda_random_starts=lambda_random_starts,
    joint_oracle=joint, verbose=True)

# ── Adaptive: IPOPT ────────────────────────────────────────────────────────
print("\n[adaptive — IPOPT (20 starts)] ...")
a_ipopt = algorithm_adaptive(
    K=K, d=d, objectives=objs, grad_objectives=grads, L=L, x0=x0,
    mode="gn",
    max_outer=max_outer_ipopt, max_inner=max_inner,
    eval_every_n_grads=eval_every_n_grads,
    target_cov=bl["cov_history"][-1],
    lambda_selector="optimize",
    lambda_max_starts=lambda_max_starts_ipopt,
    joint_oracle=joint, verbose=True)

# ── Summary ────────────────────────────────────────────────────────────────
final_err = bl["cov_history"][-1]
print(f"\n  BL     final GN* = {final_err:.4e}  "
      f"(ge={bl['grad_evals_history'][-1]}, cpu={bl['cpu_times'][-1]:.2f}s)")
print(f"  Sample final GN* = {a_sample['cov_history'][-1]:.4e}  "
      f"(ge={a_sample['grad_evals_history'][-1]}, cpu={a_sample['cpu_times'][-1]:.2f}s, "
      f"bundle={a_sample['bundle'].m})")
print(f"  IPOPT  final GN* = {a_ipopt['cov_history'][-1]:.4e}  "
      f"(ge={a_ipopt['grad_evals_history'][-1]}, cpu={a_ipopt['cpu_times'][-1]:.2f}s, "
      f"bundle={a_ipopt['bundle'].m})")

# ── Plot ───────────────────────────────────────────────────────────────────
ylabel = (r"$\sup_{\lambda\in\Delta_K} "
          r"[\min_{x_i\in\mathcal{B}_m} \|\nabla F_\lambda(x_i)\|^2]$")
fig, (ax_t, ax_g) = plt.subplots(1, 2, figsize=(12, 4.6))

for ax in (ax_t, ax_g):
    ax.axhline(final_err, color="#2ca02c", ls="--", lw=1.4,
               label=f"baseline final error = {final_err:.3e}")

ax_t.plot(a_sample["cpu_times"], a_sample["cov_history"],
          color="#1f77b4", marker="o", ms=4, lw=1.8,
          label=f"adaptive (sample, {lambda_random_starts} draws)")
ax_t.plot(a_ipopt["cpu_times"], a_ipopt["cov_history"],
          color="#ff7f0e", marker="^", ms=4, lw=1.8,
          label=f"adaptive (IPOPT, {lambda_max_starts_ipopt} starts)")

ax_g.plot(a_sample["grad_evals_history"], a_sample["cov_history"],
          color="#1f77b4", marker="o", ms=4, lw=1.8,
          label=f"adaptive (sample, {lambda_random_starts} draws)")
ax_g.plot(a_ipopt["grad_evals_history"], a_ipopt["cov_history"],
          color="#ff7f0e", marker="^", ms=4, lw=1.8,
          label=f"adaptive (IPOPT, {lambda_max_starts_ipopt} starts)")

ax_t.set_xlabel("CPU time (s)")
ax_g.set_xlabel("total gradient evaluations")
for ax in (ax_t, ax_g):
    ax.set_ylabel(ylabel)
    ax.set_yscale("log")
    ax.grid(True, which="both", alpha=0.25)
    ax.legend(frameon=False, fontsize=9)
ax_t.set_title("worst-case squared gradient norm vs CPU time")
ax_g.set_title("worst-case squared gradient norm vs gradient evals")

title = f"MLP K={K}, p={p}, n={n}, h={h}, d={d} — sample vs IPOPT"
fig.suptitle(title, fontsize=12)
fig.tight_layout(rect=(0, 0, 1, 0.96))

out_path = f"mlp_K{K}_p{p}_n{n}_h{h}_sample_vs_ipopt.png"
fig.savefig(out_path, dpi=150, bbox_inches="tight")
fig
